In [ ]:
!pip -q install ragas langchain pandas tqdm
!pip -q install langchain-community


In [ ]:
from langchain_community.chat_models import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

def load_api_key(name, fallback_names=()):
    import os
    import sys
    from pathlib import Path

    candidates = (name, *fallback_names)
    repo_root = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "common" / "__init__.py").exists()
        ),
        None,
    )

    get_api_key = None
    if repo_root is not None:
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        try:
            from common import get_api_key as shared_get_api_key
        except ImportError:
            pass
        else:
            get_api_key = shared_get_api_key

    if get_api_key is not None:
        for candidate in candidates:
            try:
                return get_api_key(candidate)
            except KeyError:
                pass

    for candidate in candidates:
        value = os.getenv(candidate, "").strip()
        if value:
            return value

    joined = ", ".join(candidates)
    raise RuntimeError(
        f"Missing API key. Set one of [{joined}] in the environment"
        " or the top-level config file."
    )

avalai_api_key = load_api_key("AVALAI_API_KEY", ("OPENAI_API_KEY",))

avalai_base_url = "https://api.avalai.ir/v1"
avalai_model = "gpt-5-nano"

# ساخت LLM با provider سفارشی
lc_llm = ChatOpenAI(
    openai_api_key=avalai_api_key,
    openai_api_base=avalai_base_url,
    model=avalai_model,
    temperature=0.0
)

ragas_llm = LangchainLLMWrapper(lc_llm)


In [ ]:
import json, os
from ragas.dataset_schema import SingleTurnSample, evaluationdataset

images_dir = "/content/drive/MyDrive/Research/Wageningen University /datasets/soil health dataset"  # مسیر فولدر عکس‌ها
json_path = "/content/drive/MyDrive/Research/Wageningen University /datasets/Output SoilHealth/rdf_extractions_fewShot.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

samples = []
for item in data.get("dataset", []):
    img_name = item.get("source_image")
    rdf_text = item.get("rdf_graph_turtle", "")
    img_path = os.path.join(images_dir, img_name)

    if not os.path.exists(img_path):
        print("⚠️ تصویر پیدا نشد:", img_path)
        continue

    sample = SingleTurnSample(
        user_input=f"Extract RDF triples from this image ({img_name})",
        response=rdf_text,
        retrieved_contexts=[img_path],  # خیلی مهم برای متریک‌های مولتی‌مودال
        metadata={"image": img_name}
    )
    samples.append(sample)

dataset = evaluationdataset(samples=samples)
print("تعداد نمونه‌ها:", len(samples))


In [ ]:
from ragas import evaluate
from ragas.metrics import MultiModalFaithfulness, MultiModalRelevance

metrics = [MultiModalFaithfulness(), MultiModalRelevance()]

results = evaluate(
    dataset,
    metrics=metrics,
    llm=ragas_llm,
    show_progress=True
)

df = results.to_pandas()
df.head()


In [ ]:
import pandas as pd

# فایل per_sample_scores را بخوان
df = pd.read_csv("/content/per_sample_scores.csv")

# محاسبه میانگین‌ها
summary = df[["faithful_rate", "relevance_rate"]].mean().reset_index()
summary.columns = ["metric", "mean_score"]

# ذخیره
summary.to_csv("/content/summary.csv", index=False)

summary
